# 잔차 패턴 분류범주 추가 실험 — Validation 최종 비교
공통 1~4는 사용자 수정본 그대로 유지했습니다. 기존 커널에서는 **공통 5부터** 실행하세요.

- PART A: Train 내부 홀드아웃에서 사용자가 지정한 임계값 비교.
- PART B: 고정 5-Fold의 중첩 회귀 OOF로 분류 예측범주 생성 후, 전체 Train 회귀 학습.
- **최종 성능 비교는 독립 Validation RMSE만 사용.** Fold별 회귀 성능 진단은 없습니다.
- PART B: 회귀 27회 + 분류 6회(단일 범주 상수 예측이면 실제 분류 fit 감소).
- 임계값은 직접 지정합니다. 분류 확률·잔차 보정 회귀를 추가하지 않습니다.


In [1]:
# 공통 1. 라이브러리

from pathlib import Path
from datetime import datetime           # 실험 실행 시각 기록
from uuid import uuid4                  # 실험별 고유 ID 생성
import hashlib                          # 데이터 및 Fold 정합성 검증용 해시
import json                             # 실험 설정값 JSON 파일 저장
import warnings                         # 경고 메시지 출력 및 관리
import platform                         # Python 실행 환경 정보 확인
import importlib.metadata               # 설치된 패키지 버전 확인

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from catboost import CatBoostRegressor, CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import mean_squared_error, f1_score, confusion_matrix

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'parqueted_data'
CONFIG_DIR = PROJECT_ROOT / 'config'
OUTPUT_ROOT = Path.cwd() / 'residual_class_experiments'

# ============================================================
# 0. 데이터 불러오기
# ============================================================

train = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '3_train_clean3.parquet')
val = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '3_val_clean3.parquet')

train_kf_oof = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '0_kf_oof_train2_index_fixed.parquet')

kf_fold_map = pd.read_csv(PROJECT_ROOT / 'config' / '0_kf_event_fold_match.csv')

In [3]:
RANDOM_STATE = 42
REG_ITERATIONS = 3000
EARLY_STOPPING_ROUNDS = 100
THREAD_COUNT = -1
CLASSES = ['under', 'normal', 'over']
NEW_FEATURE = 'pred_residual_class'  # 이번 실험에서 새로 만드는 예측범주 열
# 분류기 설정은 기존 회귀 하이퍼파라미터가 아닌 신규 실험 설정입니다.
LGB_PARAMS = dict(n_estimators=300, learning_rate=0.05, num_leaves=31,
                  random_state=42, n_jobs=THREAD_COUNT, verbosity=-1,
                  deterministic=True, force_col_wise=True)
CAT_CLS_PARAMS = dict(iterations=1000, random_state=42, loss_function='MultiClass',
                      thread_count=THREAD_COUNT, allow_writing_files=False, verbose=False)


In [4]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_t',
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]


# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_rain_1h',
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_rain_1h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols
num_cols_fe_wl = num_cols + [
    col for col in wl_diff_cols if col != 'wl_t'
]
num_cols_fe = num_cols_fe_wl + [
    col for col in rain_int_cols if col not in ['imerg_rain_1h', 'sfc_rain_1h']
]

print(f"""<변수개수 검증>
    수위변수: {len(wl_cols)}
    강우변수: {len(rain_cols)}, 
    환경변수: {len(env_cols)}, 
    수위차분(wl_t포함): {len(wl_diff_cols)},
    강우강도(rain_1h포함): {len(rain_int_cols)}
    기존 숫자열(수위 + 강우 + 환경): {len(num_cols)}, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): {len(num_cols_fe_wl)},
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): {len(num_cols_fe)}
""")


<변수개수 검증>
    수위변수: 6
    강우변수: 9, 
    환경변수: 7, 
    수위차분(wl_t포함): 6,
    강우강도(rain_1h포함): 6
    기존 숫자열(수위 + 강우 + 환경): 22, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): 27,
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): 31



In [5]:
# 공통 3. 원본은 읽기만 수행; 행 순서를 바꾸는 merge/reset_index/sort 금지
for name, df in [('train', train), ('val', val), ('기존 OOF', train_kf_oof)]:
    assert df.index.name == 'row_id', f'{name}: row_id 인덱스 필요'
    assert df.index.is_unique and not df.index.hasnans, f'{name}: 중복/결측 row_id'
    assert 'row_id' not in df.columns
assert train.index.equals(train_kf_oof.index), 'Train/기존 OOF 행 순서 불일치: 자동 정렬하지 않음'
assert train.index.intersection(val.index).empty
assert set(train['event_id']).isdisjoint(set(val['event_id']))
assert kf_fold_map.columns.tolist() == ['event_id', 'fold_num']
assert not kf_fold_map.isna().any().any() and kf_fold_map['event_id'].is_unique
assert set(kf_fold_map['fold_num']) == {1, 2, 3, 4, 5}
assert set(train['event_id']) == set(kf_fold_map['event_id'])
assert not train['event_id'].isna().any() and not val['event_id'].isna().any()

kf_groups = train['event_id']
kf_fold_num = kf_groups.map(kf_fold_map.set_index('event_id')['fold_num']).astype(int)
assert kf_fold_num.notna().all()
assert np.array_equal(kf_fold_num.to_numpy(), train_kf_oof['fold_num'].to_numpy())
assert (kf_fold_num.groupby(kf_groups).nunique() == 1).all()
assert np.array_equal(train[target_col].to_numpy(), train_kf_oof['y_kf_val'].to_numpy())
# 기존 OOF의 원래 입력 열도 현재 Train과 완전히 같은지 확인
pd.testing.assert_frame_equal(train, train_kf_oof[train.columns])
assert np.isfinite(train_kf_oof[['y_kf_pred', 'resid', 'resid_sq']].to_numpy()).all()
assert np.allclose(train_kf_oof['resid'], train_kf_oof['y_kf_val'] - train_kf_oof['y_kf_pred'])
assert np.allclose(train_kf_oof['resid_sq'], train_kf_oof['resid'] ** 2)

X_train_with_station = train[num_cols_fe + ['station_id']].copy()
X_val_with_station = val[num_cols_fe + ['station_id']].copy()
for X in [X_train_with_station, X_val_with_station]:
    assert X['station_id'].notna().all()
    X['station_id'] = X['station_id'].astype(str)
    assert not np.isinf(X[num_cols_fe].to_numpy(dtype=float)).any()
    assert all(pd.api.types.is_numeric_dtype(X[c]) for c in num_cols_fe)
y_train = train[target_col]
assert np.isfinite(y_train.to_numpy()).all()
assert len(num_cols_fe) == 31 and X_train_with_station.shape[1] == 32

# 원본 확인용 스냅샷: 실험 종료 때 다시 비교
original_row_index = train.index.copy()
original_fold_num = kf_fold_num.copy()
old_oof_rmse = float(np.sqrt(mean_squared_error(y_train, train_kf_oof['y_kf_pred'])))
print(f'Train={train.shape}, Validation={val.shape}, 고정 Fold=5')
print(f'저장된 기존 OOF RMSE={old_oof_rmse:.10f} (새 평가의 기준값으로 대입하지 않음)')
display(pd.crosstab(kf_fold_num, columns='행 수'))


Train=(616293, 35), Validation=(133316, 35), 고정 Fold=5
저장된 기존 OOF RMSE=0.2328268789 (새 평가의 기준값으로 대입하지 않음)


col_0,행 수
event_id,
1,122996
2,123399
3,123399
4,123216
5,123283


## 공통 학습·검증 함수
반복되는 OOF 생성과 누출 검증만 함수로 묶었습니다. A/B의 임계값·외부 Fold 반복은 각 PART에 직접 표시합니다.
`residual_label`은 분류 정답이고, `pred_residual_class`는 분류기의 `predict()` 결과입니다.

내부 OOF는 해당 행의 타깃이 트리 학습에는 들어가지 않지만 **조기 종료 선택에는 쓰입니다**.
이는 이번에 지정한 내부 학습 방식입니다. 외부 평가 데이터와 독립 Validation에는 적용하지 않습니다.


In [6]:
# 공통 4. 라벨/평가지표/실행 기록: 임계값을 선정하는 코드는 없음

def residual_labels(residual, threshold): # 잔차와 임계값을 비교해 under, normal, over 3개 정답 라벨 생성
    assert np.isfinite(threshold) and threshold > 0, '임계값은 양의 유한값이어야 함'
    assert np.isfinite(residual.to_numpy()).all()
    return pd.Series(np.where(residual > threshold, 'under',
                     np.where(residual < -threshold, 'over', 'normal')),
                     index=residual.index, name='residual_label')


def rmse(y, pred):  # 실제값과 예측값의 RMSE 계산
    assert y.index.equals(pred.index) and np.isfinite(pred.to_numpy()).all()
    return float(np.sqrt(mean_squared_error(y, pred)))


def macro_f1(y, pred):  # 3개 분류범주의 Macro F1 계산
    assert y.index.equals(pred.index)
    return float(f1_score(y, pred, labels=CLASSES, average='macro', zero_division=0))


def class_summary(labels):  # 분류범주별 데이터 개수와 비율 계산
    count = labels.value_counts().reindex(CLASSES, fill_value=0)
    return pd.DataFrame({'count': count, 'ratio': count / len(labels)})


def index_hash(index):  # 인덱스 정보를 SHA-256 해시로 변환해 정합성 기록
    return hashlib.sha256(pd.util.hash_pandas_object(index).to_numpy().tobytes()).hexdigest()


def start_run(part, settings):  # 실험별 결과 폴더 생성 및 설정값을 settings.json에 저장
    folder = OUTPUT_ROOT / (part + '_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid4().hex[:8])
    folder.mkdir(parents=True, exist_ok=False)
    manifest = dict(part=part, settings=settings, reg_iterations=REG_ITERATIONS,
                    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
                    lgb_params=LGB_PARAMS, cat_cls_params=CAT_CLS_PARAMS,
                    files={
                        'train': str(PROJECT_ROOT / 'parqueted_data' / '3_train_clean3.parquet'),
                        'val': str(PROJECT_ROOT / 'parqueted_data' / '3_val_clean3.parquet'),
                        'oof': str(PROJECT_ROOT / 'parqueted_data' / '0_kf_oof_train2_index_fixed.parquet'),
                        'fold': str(PROJECT_ROOT / 'config' / '0_kf_event_fold_match.csv')
                    },
                    train_index_hash=index_hash(original_row_index),
                    fold_hash=index_hash(pd.Index(original_fold_num.to_numpy())),
                    old_oof_rmse=old_oof_rmse,
                    versions={p: importlib.metadata.version(p) for p in
                              ['numpy', 'pandas', 'pyarrow', 'scikit-learn', 'catboost', 'lightgbm']},
                    python=platform.python_version())
    (folder / 'settings.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
    return folder


# 학습·검증·예측 데이터가 중복되거나 금지된 데이터를 사용하는지 검사
def audit(stage, fit_ids, eval_ids, pred_ids, forbidden_ids, log,
          label_source_ids=None, internal_regression=False):
    # forbidden_ids는 외부 검증 및 독립 Validation 행; 중첩 단계는 예측대상 내부 Fold도 추가
    for ids in [fit_ids, eval_ids, pred_ids]:
        assert ids.is_unique
    assert fit_ids.intersection(eval_ids).empty
    assert fit_ids.intersection(pred_ids).empty, f'{stage}: In-sample 예측 금지'
    assert fit_ids.intersection(forbidden_ids).empty
    assert eval_ids.intersection(forbidden_ids).empty, f'{stage}: 외부 정답 eval_set 사용 금지'
    if not internal_regression:
        assert eval_ids.empty, f'{stage}: 내부 잔차 회귀 외 eval_set 금지'
    if label_source_ids is not None:
        assert label_source_ids.intersection(forbidden_ids).empty
        assert label_source_ids.intersection(pred_ids).empty, f'{stage}: 분류 정답에 예측 Fold 간접 포함'
        assert fit_ids.difference(label_source_ids).empty
    log.append(dict(stage=stage, fit_n=len(fit_ids), eval_n=len(eval_ids), predict_n=len(pred_ids),
                    fit_hash=index_hash(fit_ids), eval_hash=index_hash(eval_ids), predict_hash=index_hash(pred_ids),
                    forbidden_hash=index_hash(forbidden_ids),
                    label_source_hash=index_hash(label_source_ids) if label_source_ids is not None else None,
                    checks_passed=True))


def check_original():  # 실험 과정에서 Train 행 순서와 고정 Fold가 변경되지 않았는지 검사
    assert train.index.equals(original_row_index)
    pd.testing.assert_series_equal(kf_fold_num, original_fold_num)
    assert np.array_equal(train['event_id'].map(kf_fold_map.set_index('event_id')['fold_num']),
                          original_fold_num.to_numpy())


In [7]:
# 공통 5. 내부 회귀 OOF: 전달된 데이터 안의 고정 Fold만 사용

def regression_oof(X, y, folds, forbidden_ids, tag, log):  # fold별 회귀 OOF 예측값 생성
    assert X.index.equals(y.index) and X.index.equals(folds.index)
    assert X.index.intersection(forbidden_ids).empty
    assert folds.nunique() >= 2 and folds.notna().all()
    pred = pd.Series(np.nan, index=X.index, name='regression_oof')
    seen = pd.Series(0, index=X.index)
    iterations = []
    for inner_fold in sorted(folds.unique()):
        fit_ids = X.index[folds.to_numpy() != inner_fold]
        eval_ids = X.index[folds.to_numpy() == inner_fold]
        audit(f'{tag}/reg_inner_{inner_fold}', fit_ids, eval_ids, eval_ids,
              forbidden_ids, log, internal_regression=True)
        model = CatBoostRegressor(random_state=RANDOM_STATE, iterations=REG_ITERATIONS,
                    cat_features=['station_id'], thread_count=THREAD_COUNT, allow_writing_files=False)
        model.fit(X.loc[fit_ids], y.loc[fit_ids], eval_set=(X.loc[eval_ids], y.loc[eval_ids]),
                  early_stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False)
        pred.loc[eval_ids] = model.predict(X.loc[eval_ids])
        seen.loc[eval_ids] += 1
        iterations.append({'stage': tag, 'fold': int(inner_fold),
                           'best_iteration': int(model.best_iteration_), 'tree_count': int(model.tree_count_)})
        print(f'  {tag}: 내부 Fold {inner_fold}, trees={model.tree_count_}', flush=True)
    assert (seen == 1).all() and pred.notna().all()
    assert pred.index.equals(X.index)
    return pred, iterations


# PART A 전용: 홀드아웃 학습용 분류 OOF와 홀드아웃 분류기 정답 준비 
def prepare_classification_residuals(X, y, folds, forbidden_ids, tag, log):
    # 전체 학습 풀의 잔차: 외부 대상 예측용 분류기 정답
    pred, rounds = regression_oof(X, y, folds, forbidden_ids, tag + '/all_labels', log)
    residual = (y - pred).rename('residual')
    blocks = []
    # 학습용 분류 OOF는 예측 대상 Fold를 잔차 생성에서도 제외해야 함
    for class_fold in sorted(folds.unique()):
        fit_ids = X.index[folds.to_numpy() != class_fold]
        predict_ids = X.index[folds.to_numpy() == class_fold]
        protected = forbidden_ids.union(predict_ids, sort=False)
        subpred, subrounds = regression_oof(X.loc[fit_ids], y.loc[fit_ids], folds.loc[fit_ids],
                                            protected, f'{tag}/class_hold_{class_fold}', log)
        blocks.append((fit_ids, predict_ids, (y.loc[fit_ids] - subpred).rename('residual')))
        rounds.extend(subrounds)
    return residual, blocks, rounds


In [ ]:
# 공통 6. 분류기는 최종 범주만 반환; 희소/단일 범주 상황도 명시적으로 기록

# LightGBM 또는 CatBoost 분류기를 학습하고 under, normal, over 예측범주 반환
def predict_categories(X_fit, labels, X_predict, kind, forbidden_ids, label_source_ids, tag, log):
    assert X_fit.index.equals(labels.index) and labels.name == 'residual_label'
    assert set(labels.unique()).issubset(CLASSES)
    audit(tag, X_fit.index, pd.Index([]), X_predict.index, forbidden_ids, log,
          label_source_ids=label_source_ids)
    unique = labels.unique()
    if len(unique) == 1:
        # 학습할 범주가 하나뿐이면 상수 범주 예측: 실패를 숨기지 않고 기록
        warnings.warn(f'{tag}: 학습 정답이 {unique[0]} 하나뿐입니다. 상수 예측 사용.')
        log[-1]['constant_fallback'] = True
        result = np.repeat(unique[0], len(X_predict))
    else:
        if len(unique) < 3:
            warnings.warn(f'{tag}: 학습 정답에 없는 범주={set(CLASSES) - set(unique)}')
        log[-1]['constant_fallback'] = False
        if kind == 'lightgbm':
            X_fit = X_fit.copy()
            X_predict = X_predict.copy()
            # 범주 사전은 분류 학습셋에서만 생성; 미관측 관측소는 결측 범주로 처리
            station_dtype = pd.CategoricalDtype(categories=pd.unique(X_fit['station_id']), ordered=False)
            X_fit['station_id'] = X_fit['station_id'].astype(station_dtype)
            X_predict['station_id'] = X_predict['station_id'].astype(station_dtype)
            model = LGBMClassifier(**LGB_PARAMS)
            model.fit(X_fit, labels, categorical_feature=['station_id'])
        else:
            assert kind == 'catboost'
            model = CatBoostClassifier(**CAT_CLS_PARAMS, cat_features=['station_id'])
            model.fit(X_fit, labels)
        result = np.asarray(model.predict(X_predict)).reshape(-1).astype(str)
    pred = pd.Series(result, index=X_predict.index, name=NEW_FEATURE)
    assert pred.notna().all() and set(pred.unique()).issubset(CLASSES)
    return pred

# Fold별 분류모델의 OOF 예측 결과를 합쳐 Train 전체의 신규 변수 생성
def training_category_oof(X, blocks, threshold, kind, forbidden_ids, tag, log):
    pred = pd.Series(index=X.index, dtype=object, name=NEW_FEATURE)
    seen = pd.Series(0, index=X.index)
    for j, (fit_ids, predict_ids, residual) in enumerate(blocks, start=1):
        labels = residual_labels(residual, threshold)
        protected = forbidden_ids.union(predict_ids, sort=False)
        pred.loc[predict_ids] = predict_categories(X.loc[fit_ids], labels, X.loc[predict_ids], kind,
                   protected, fit_ids, f'{tag}/class_oof_{j}', log)
        seen.loc[predict_ids] += 1
    assert (seen == 1).all() and pred.notna().all() and pred.index.equals(X.index)
    return pred


def add_category(X, predicted):  # 기존 독립변수에 분류결과 담은 예측컬럼 1개 추가
    assert predicted.name == NEW_FEATURE, '분류 정답 라벨은 본 회귀 입력으로 사용할 수 없음'
    assert predicted.index.equals(X.index) and predicted.notna().all()
    assert set(predicted.unique()).issubset(CLASSES) and NEW_FEATURE not in X.columns
    result = X.assign(**{NEW_FEATURE: predicted.astype(str)})
    assert result.shape[1] == X.shape[1] + 1
    return result

# 최종 CatBoost 회귀모델 학습 및 최종 Validation 예측
def fit_main_regression(X_fit, y_fit, X_predict, forbidden_ids, tag, log):
    assert X_fit.index.equals(y_fit.index)
    assert X_fit.columns.equals(X_predict.columns)
    audit(tag, X_fit.index, pd.Index([]), X_predict.index, forbidden_ids, log)
    cat_features = ['station_id'] + ([NEW_FEATURE] if NEW_FEATURE in X_fit.columns else [])
    model = CatBoostRegressor(random_state=RANDOM_STATE, iterations=REG_ITERATIONS,
                cat_features=cat_features, thread_count=THREAD_COUNT, allow_writing_files=False,
                use_best_model=False)
    model.fit(X_fit, y_fit, verbose=False)
    return pd.Series(model.predict(X_predict), index=X_predict.index, name='regression_prediction')


## PART A — 임계값 탐색 (Train 내부 홀드아웃)
공통 셀만 실행한 상태에서 시작합니다. 독립 Validation의 타깃은 사용하지 않습니다.
잔차 계산은 임계값과 무관하므로 한 번 준비하고, **각 임계값 반복문 안에서 분류기 학습부터 회귀 RMSE 평가까지 수행**합니다.
표의 범주 비율은 홀드아웃 학습 부분의 회귀 OOF 잔차 정답 분포입니다. Macro F1은 홀드아웃에서 기준회귀 잔차로 정의한 정답과 비교합니다.


In [ ]:
# PART A 1. 사용자가 직접 후보를 입력: 예시값이며 자동 선정하지 않음
thresholds = [0.3, 0.5, 0.7, 1.0]
A_HOLDOUT_SIZE = 0.2
A_INNER_SPLITS = 4
assert len(thresholds) > 0 and len(set(thresholds)) == len(thresholds)
assert all(np.isfinite(t) and t > 0 for t in thresholds)
a_dir = start_run('PART_A', dict(thresholds=thresholds, holdout_size=A_HOLDOUT_SIZE,
                                inner_splits=A_INNER_SPLITS, random_state=RANDOM_STATE))
a_log = []


In [ ]:
# PART A 2. 모든 임계값에서 동일한 강우사상 홀드아웃/내부 Fold를 재사용
splitter = GroupShuffleSplit(n_splits=1, test_size=A_HOLDOUT_SIZE, random_state=RANDOM_STATE)
a_train_pos, a_hold_pos = next(splitter.split(X_train_with_station, y_train, groups=kf_groups))
a_train_ids = train.index[a_train_pos]
a_hold_ids = train.index[a_hold_pos]
assert set(kf_groups.loc[a_train_ids]).isdisjoint(set(kf_groups.loc[a_hold_ids]))
a_forbidden = a_hold_ids.union(val.index, sort=False)
a_folds = pd.Series(0, index=a_train_ids, dtype=int)
for fold, (_, valid_pos) in enumerate(GroupKFold(n_splits=A_INNER_SPLITS).split(
        X_train_with_station.loc[a_train_ids], y_train.loc[a_train_ids],
        groups=kf_groups.loc[a_train_ids]), start=1):
    a_folds.iloc[valid_pos] = fold
assert (a_folds > 0).all()
assert (a_folds.groupby(kf_groups.loc[a_train_ids]).nunique() == 1).all()
pd.DataFrame({'event_id': kf_groups, 'is_holdout': train.index.isin(a_hold_ids),
              'inner_fold': a_folds.reindex(train.index)}).to_csv(a_dir / 'split.csv', index=True)

# 임계값과 무관한 회귀 잔차 준비: 4 + 4*3 = 16회 회귀 학습
try:
    a_residual, a_blocks, a_rounds = prepare_classification_residuals(
        X_train_with_station.loc[a_train_ids], y_train.loc[a_train_ids], a_folds,
        a_forbidden, 'A', a_log)
    a_base_pred = fit_main_regression(X_train_with_station.loc[a_train_ids], y_train.loc[a_train_ids],
        X_train_with_station.loc[a_hold_ids], a_forbidden, 'A/baseline', a_log)
    a_base_rmse = rmse(y_train.loc[a_hold_ids], a_base_pred)
    # 홀드아웃 정답은 평가에만 사용; 분류 학습 함수로 전달하지 않음
    a_eval_residual = y_train.loc[a_hold_ids] - a_base_pred
    pd.DataFrame(a_rounds).to_csv(a_dir / 'inner_iterations.csv', index=False)
finally:
    pd.DataFrame(a_log).to_csv(a_dir / 'audit.csv', index=False)
print(f'PART A 기준 홀드아웃 RMSE: {a_base_rmse:.8f}')


In [ ]:
# PART A 3. 임계값별 분류 학습 → 범주 추가 → 회귀 평가를 같은 반복문에서 수행
a_results = []

for threshold in thresholds:
    print(f'\nPART A threshold={threshold}', flush=True)
    try:
        # 1~2. 회귀 OOF 잔차로 분류 정답과 학습 범주 비율 생성
        a_labels = residual_labels(a_residual, threshold)
        a_distribution = class_summary(a_labels)
        display(a_distribution)
        # 3~6. 학습용 분류 OOF와 홀드아웃 예측범주: LightGBM 4+1회
        a_train_category = training_category_oof(X_train_with_station.loc[a_train_ids], a_blocks,
            threshold, 'lightgbm', a_forbidden, f'A/t={threshold}', a_log)
        a_hold_category = predict_categories(X_train_with_station.loc[a_train_ids], a_labels,
            X_train_with_station.loc[a_hold_ids], 'lightgbm', a_forbidden, a_train_ids,
            f'A/t={threshold}/holdout_classifier', a_log)
        a_f1 = macro_f1(residual_labels(a_eval_residual, threshold), a_hold_category)
        # 7~10. 예측범주 1개 추가 후 동일한 기준 회귀 설정으로 학습/평가
        a_exp_pred = fit_main_regression(
            add_category(X_train_with_station.loc[a_train_ids], a_train_category),
            y_train.loc[a_train_ids],
            add_category(X_train_with_station.loc[a_hold_ids], a_hold_category),
            a_forbidden, f'A/t={threshold}/experiment', a_log)
        a_exp_rmse = rmse(y_train.loc[a_hold_ids], a_exp_pred)
        # 11. 입력 순서 그대로 누적: 최소 RMSE 임계값을 선택/확정하지 않음
        a_results.append({'threshold': threshold,
            'under_ratio': a_distribution.loc['under', 'ratio'],
            'normal_ratio': a_distribution.loc['normal', 'ratio'],
            'over_ratio': a_distribution.loc['over', 'ratio'],
            'macro_f1': a_f1, 'baseline_rmse': a_base_rmse,
            'experiment_rmse': a_exp_rmse, 'delta_rmse': a_exp_rmse - a_base_rmse})
        pd.DataFrame(a_results).to_csv(a_dir / 'threshold_results.csv', index=False)
        a_distribution.to_csv(a_dir / f'class_counts_{threshold}.csv')
        pd.DataFrame({'baseline_pred': a_base_pred, 'experiment_pred': a_exp_pred,
                      NEW_FEATURE: a_hold_category}).to_parquet(a_dir / f'holdout_{threshold}.parquet')
        print(a_results[-1], flush=True)
    except Exception as exc:
        (a_dir / 'error.txt').write_text(f'threshold={threshold}\n{type(exc).__name__}: {exc}', encoding='utf-8')
        raise
    finally:
        pd.DataFrame(a_log).to_csv(a_dir / 'audit.csv', index=False)
check_original()
a_results_df = pd.DataFrame(a_results)
display(a_results_df.rename(columns={'threshold':'임계값', 'under_ratio':'과소예측 비율',
    'normal_ratio':'정상예측 비율', 'over_ratio':'과대예측 비율', 'macro_f1':'Macro F1',
    'baseline_rmse':'기준 RMSE', 'experiment_rmse':'실험 RMSE', 'delta_rmse':'ΔRMSE'}))


In [ ]:
# PART A 4. 임계값 순서를 유지한 비교 그래프: 선정/추천 임계값 표시 없음
fig, ax = plt.subplots(figsize=(8, 4))
positions = np.arange(len(a_results_df))
ax.plot(positions, a_results_df['experiment_rmse'], marker='o', label='With predicted class')
ax.axhline(a_base_rmse, linestyle='--', color='gray', label='Baseline: same holdout')
ax.set_xticks(positions, [str(t) for t in thresholds])
ax.set(xlabel='Residual threshold (user supplied)', ylabel='Holdout RMSE', title='PART A: threshold comparison')
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(a_dir / 'threshold_rmse.png', dpi=160)
plt.show()
print('PART A 완료. 결과를 확인한 뒤 PART B의 final_threshold를 직접 입력하세요.')


## PART B — 분류 OOF 생성 및 독립 Validation 최종 비교
공통 준비 후 PART B만 실행할 수 있습니다. PART A의 변수/모델을 읽지 않습니다.

예측 대상 Fold A를 제외한 B·C·D·E에서 내부 4-Fold 회귀 OOF 잔차를 만들고,
그 잔차로 CatBoost 분류기를 학습해 A의 범주를 예측합니다. 이를 5회 반복합니다.
**여기서는 Fold별 본 회귀 학습·RMSE 계산을 하지 않습니다.**

합친 Train 분류 OOF는 전체 Train 최종 회귀 입력으로만 사용합니다.
Validation용 분류기는 Train 전체의 별도 고정 5-Fold 회귀 OOF 잔차로 학습합니다.
Validation 타깃은 마지막 평가에만 사용합니다.


In [ ]:
# PART B 1. 확정 임계값 직접 입력; A 결과에서 자동 선택하지 않음
final_threshold = None  # 예: 1.0. 사용자가 직접 확정한 값으로 변경
if final_threshold is None:
    raise ValueError('final_threshold에 사용자가 확정한 양의 임계값을 입력하세요.')
assert np.isfinite(final_threshold) and final_threshold > 0
b_dir = start_run('PART_B', dict(final_threshold=float(final_threshold), classification_oof_folds=5,
                                final_metric='independent_validation_rmse'))
b_log = []
b_rounds = []
b_oof_category = pd.Series(index=train.index, dtype=object, name=NEW_FEATURE)
b_seen = pd.Series(0, index=train.index)


In [ ]:
# PART B 2. 중첩 분류 OOF만 생성: 회귀 5×4=20회, 분류 5회
for fold_num in [1, 2, 3, 4, 5]:
    print(f'\nPART B 분류 OOF Fold {fold_num}/5', flush=True)
    b_train_ids = train.index[kf_fold_num.to_numpy() != fold_num]
    b_predict_ids = train.index[kf_fold_num.to_numpy() == fold_num]
    b_forbidden = b_predict_ids.union(val.index, sort=False)
    assert set(kf_groups.loc[b_train_ids]).isdisjoint(set(kf_groups.loc[b_predict_ids]))
    assert kf_fold_num.loc[b_train_ids].nunique() == 4
    try:
        # 예측 대상 Fold는 내부 회귀 fit/eval_set 및 분류 정답 생성 모두에서 제외
        b_inner_pred, rounds = regression_oof(
            X_train_with_station.loc[b_train_ids], y_train.loc[b_train_ids],
            kf_fold_num.loc[b_train_ids], b_forbidden, f'B/class_fold_{fold_num}', b_log)
        b_rounds.extend(rounds)
        b_labels = residual_labels(y_train.loc[b_train_ids] - b_inner_pred, final_threshold)
        display(class_summary(b_labels))
        b_predicted_category = predict_categories(
            X_train_with_station.loc[b_train_ids], b_labels,
            X_train_with_station.loc[b_predict_ids], 'catboost', b_forbidden, b_train_ids,
            f'B/class_fold_{fold_num}/classifier', b_log)
        assert b_predicted_category.index.equals(b_predict_ids)
        b_oof_category.loc[b_predict_ids] = b_predicted_category
        b_seen.loc[b_predict_ids] += 1
        # 분류 OOF 진행상태만 저장. Fold별 회귀 예측/성능표는 생성하지 않음
        pd.DataFrame({'fold_num': kf_fold_num, NEW_FEATURE: b_oof_category,
                      'completed': b_seen == 1}).to_parquet(b_dir / 'classification_oof.parquet')
        pd.DataFrame(b_rounds).to_csv(b_dir / 'inner_iterations.csv', index=False)
    except Exception as exc:
        (b_dir / 'error.txt').write_text(
            f'classification_fold={fold_num}\n{type(exc).__name__}: {exc}', encoding='utf-8')
        raise
    finally:
        pd.DataFrame(b_log).to_csv(b_dir / 'audit.csv', index=False)
assert (b_seen == 1).all()
assert b_oof_category.notna().all() and b_oof_category.index.equals(train.index)
assert set(b_oof_category.unique()).issubset(CLASSES)
check_original()
print('Train 분류 OOF 생성 완료. 다음 셀에서 독립 Validation 최종 비교를 수행합니다.')


### 독립 Validation 최종 평가
Train 전체 고정 5-Fold 회귀 OOF(5회) → 최종 분류기(1회) → Validation 예측범주.
전체 Train 기준·실험 회귀(각 1회)는 3000회 고정 학습하며 eval_set을 사용하지 않습니다.
Train 실험 입력에는 앞에서 만든 `b_oof_category`만 붙입니다.

회귀 OOF는 분류 정답 생성용이며 별도의 회귀 OOF 성능 진단을 수행하지 않습니다.


In [ ]:
# PART B 3. 독립 Validation에는 어떤 모델도 fit/eval_set 하지 않음
try:
    b_full_reg_oof, b_full_rounds = regression_oof(X_train_with_station, y_train, kf_fold_num,
                                                  val.index, 'B/full_train_labels', b_log)
    b_full_labels = residual_labels(y_train - b_full_reg_oof, final_threshold)
    b_val_category = predict_categories(X_train_with_station, b_full_labels, X_val_with_station,
                        'catboost', val.index, train.index, 'B/final_classifier', b_log)
    b_final_base_pred = fit_main_regression(X_train_with_station, y_train, X_val_with_station,
                                           val.index, 'B/final_baseline', b_log)
    b_final_exp_pred = fit_main_regression(add_category(X_train_with_station, b_oof_category), y_train,
                        add_category(X_val_with_station, b_val_category), val.index, 'B/final_experiment', b_log)
    # 여기서 처음 독립 Validation 정답을 읽어 최종 평가
    y_val = val[target_col]
    assert np.isfinite(y_val.to_numpy()).all()
    b_final_base_rmse = rmse(y_val, b_final_base_pred)
    b_final_exp_rmse = rmse(y_val, b_final_exp_pred)
    b_val_true_labels = residual_labels(y_val - b_final_base_pred, final_threshold)
    b_final_summary = pd.DataFrame([dict(
        final_threshold=final_threshold, baseline_rmse=b_final_base_rmse,
        experiment_rmse=b_final_exp_rmse, delta_rmse=b_final_exp_rmse - b_final_base_rmse,
        improvement_pct=(b_final_base_rmse - b_final_exp_rmse) / b_final_base_rmse * 100
                            if b_final_base_rmse else np.nan,
        macro_f1=macro_f1(b_val_true_labels, b_val_category))])
    display(b_final_summary)
    b_final_summary.to_csv(b_dir / 'FINAL_validation_summary.csv', index=False)
    pd.DataFrame({'y_true': y_val, 'baseline_pred': b_final_base_pred,
                  'experiment_pred': b_final_exp_pred, NEW_FEATURE: b_val_category,
                  'residual_label_evaluation_only': b_val_true_labels}).to_parquet(
                      b_dir / 'FINAL_validation_predictions.parquet')
    pd.DataFrame(confusion_matrix(b_val_true_labels, b_val_category, labels=CLASSES),
                 index=CLASSES, columns=CLASSES).to_csv(b_dir / 'validation_confusion_matrix.csv')
    pd.DataFrame(b_rounds + b_full_rounds).to_csv(b_dir / 'inner_iterations.csv', index=False)
    check_original()
    print('최종 평가 완료: ΔRMSE < 0이면 개선. 임계값 자동 변경 없음.')
except Exception as exc:
    (b_dir / 'error.txt').write_text(f'final_validation\n{type(exc).__name__}: {exc}', encoding='utf-8')
    raise
finally:
    pd.DataFrame(b_log).to_csv(b_dir / 'audit.csv', index=False)


## 실행 결과와 학습 횟수
- 최종 지표: `FINAL_validation_summary.csv`. ΔRMSE=실험−기준, 음수이면 개선.
- 분류 Macro F1과 혼동행렬은 Validation 참고 지표입니다. 최종 판단은 RMSE입니다.
- Train 분류 OOF: `classification_oof.parquet`, row_id 인덱스 유지.
- 회귀 Fold별 RMSE/OOF RMSE 및 해당 결과 파일은 생성하지 않습니다.
- PART A 후보 m개: 회귀 17+m회, LightGBM 분류 5m회.
- PART B: Train 분류 OOF용 회귀 20 + Validation 분류 정답용 회귀 5 + 최종 회귀 2 = 27회.
  CatBoost 분류는 Train OOF 5 + Validation용 1 = 6회.
- 내부 회귀만 조기 종료 100회 적용. 외부 예측 대상 Fold/Validation을 eval_set으로 사용하지 않습니다.
- 단일 범주 상수 예측은 경고와 audit 기록으로 표시합니다.
- `audit.csv`, `inner_iterations.csv`, `settings.json`, 최종 예측/혼동행렬은 유지합니다.
- 원본 데이터는 변경하지 않으며 Test 데이터는 읽지 않습니다.
